In [1]:
"""
Algoritmo Genético para el Problema de la Mochila (Knapsack)
--------------------------------------------------------------
Basado en la Tabla 3.1: 8 artículos, capacidad máxima = 25 kg.
Cada individuo es un cromosoma binario de 8 bits:
    bit = 1 -> se lleva el artículo
    bit = 0 -> no se lleva el artículo

Objetivo: maximizar el valor (utilidad) total sin exceder el peso máximo.
"""

import random

# ---------------------------------------------------------
# 1. Datos del problema (Tabla 3.1)
# ---------------------------------------------------------
articulos = [
    {"id": "B",  "desc": "Botella de agua",        "peso": 3, "valor": 8},
    {"id": "L",  "desc": "Linterna",                "peso": 2, "valor": 7},
    {"id": "K",  "desc": "Kit de primeros auxilios", "peso": 4, "valor": 11},
    {"id": "R",  "desc": "Ropa extra",              "peso": 5, "valor": 6},
    {"id": "C",  "desc": "Comida enlatada",         "peso": 7, "valor": 16},
    {"id": "S",  "desc": "Saco de dormir",          "peso": 6, "valor": 12},
    {"id": "M",  "desc": "Mapa y brújula",          "peso": 1, "valor": 7},
    {"id": "Cu", "desc": "Cuerda de escalada",       "peso": 8, "valor": 18},
]

PESO_MAXIMO = 25
N_GENES = len(articulos)          # 8 genes -> 256 combinaciones posibles

# ---------------------------------------------------------
# 2. Parámetros del AG
# ---------------------------------------------------------
TAM_POBLACION = 20
N_GENERACIONES = 100
PROB_CRUCE = 0.8
PROB_MUTACION = 0.05
TAM_TORNEO = 3
ELITISMO = 2          # número de mejores individuos que pasan directo


# ---------------------------------------------------------
# 3. Funciones del AG
# ---------------------------------------------------------
def crear_individuo():
    """Cromosoma binario aleatorio de longitud N_GENES."""
    return [random.randint(0, 1) for _ in range(N_GENES)]


def peso_valor(individuo):
    """Calcula el peso total y el valor total de un individuo."""
    peso_total = sum(g * a["peso"] for g, a in zip(individuo, articulos))
    valor_total = sum(g * a["valor"] for g, a in zip(individuo, articulos))
    return peso_total, valor_total


def fitness(individuo):
    """
    Valor total si el peso no excede el máximo.
    Si excede, se penaliza fuertemente para descartar la solución.
    """
    peso_total, valor_total = peso_valor(individuo)
    if peso_total > PESO_MAXIMO:
        exceso = peso_total - PESO_MAXIMO
        return max(0, valor_total - exceso * 10)  # penalización
    return valor_total


def seleccion_torneo(poblacion, fitnesses):
    """Selecciona un individuo por torneo de tamaño TAM_TORNEO."""
    participantes = random.sample(list(zip(poblacion, fitnesses)), TAM_TORNEO)
    ganador = max(participantes, key=lambda x: x[1])
    return ganador[0][:]  # copia


def cruce_un_punto(padre1, padre2):
    """Cruce de un punto."""
    if random.random() > PROB_CRUCE:
        return padre1[:], padre2[:]
    punto = random.randint(1, N_GENES - 1)
    hijo1 = padre1[:punto] + padre2[punto:]
    hijo2 = padre2[:punto] + padre1[punto:]
    return hijo1, hijo2


def mutacion(individuo):
    """Mutación bit a bit (flip) con probabilidad PROB_MUTACION."""
    for i in range(len(individuo)):
        if random.random() < PROB_MUTACION:
            individuo[i] = 1 - individuo[i]
    return individuo


def describir(individuo):
    """Lista legible de artículos seleccionados."""
    return [a["id"] for g, a in zip(individuo, articulos) if g == 1]


# ---------------------------------------------------------
# 4. Ciclo evolutivo
# ---------------------------------------------------------
def ejecutar_ag():
    poblacion = [crear_individuo() for _ in range(TAM_POBLACION)]
    mejor_historico = None
    mejor_fitness_historico = -1

    for gen in range(N_GENERACIONES):
        fitnesses = [fitness(ind) for ind in poblacion]

        # Guardar el mejor de todos los tiempos
        for ind, fit in zip(poblacion, fitnesses):
            if fit > mejor_fitness_historico:
                mejor_fitness_historico = fit
                mejor_historico = ind[:]

        # Elitismo: ordenar población por fitness descendente
        pob_ordenada = [ind for _, ind in sorted(
            zip(fitnesses, poblacion), key=lambda x: x[0], reverse=True)]
        nueva_poblacion = pob_ordenada[:ELITISMO]

        # Generar el resto de la nueva población
        while len(nueva_poblacion) < TAM_POBLACION:
            padre1 = seleccion_torneo(poblacion, fitnesses)
            padre2 = seleccion_torneo(poblacion, fitnesses)
            hijo1, hijo2 = cruce_un_punto(padre1, padre2)
            nueva_poblacion.append(mutacion(hijo1))
            if len(nueva_poblacion) < TAM_POBLACION:
                nueva_poblacion.append(mutacion(hijo2))

        poblacion = nueva_poblacion

        if gen % 10 == 0 or gen == N_GENERACIONES - 1:
            peso_t, valor_t = peso_valor(mejor_historico)
            print(f"Gen {gen:3d} | Mejor valor: {valor_t:3d} | "
                  f"Peso: {peso_t:2d}/{PESO_MAXIMO} | "
                  f"Artículos: {describir(mejor_historico)}")

    return mejor_historico


# ---------------------------------------------------------
# 5. Ejecución y resultado final
# ---------------------------------------------------------
if __name__ == "__main__":
    random.seed()  # quitar/fijar semilla si se quiere reproducibilidad
    mejor = ejecutar_ag()
    peso_final, valor_final = peso_valor(mejor)

    print("\n=== RESULTADO FINAL ===")
    print(f"Cromosoma óptimo: {mejor}")
    print(f"Artículos seleccionados: {describir(mejor)}")
    print(f"Peso total: {peso_final} / {PESO_MAXIMO} kg")
    print(f"Valor total (utilidad): {valor_final}")

Gen   0 | Mejor valor:  58 | Peso: 25/25 | Artículos: ['K', 'R', 'C', 'M', 'Cu']
Gen  10 | Mejor valor:  67 | Peso: 25/25 | Artículos: ['B', 'L', 'K', 'C', 'M', 'Cu']
Gen  20 | Mejor valor:  67 | Peso: 25/25 | Artículos: ['B', 'L', 'K', 'C', 'M', 'Cu']
Gen  30 | Mejor valor:  67 | Peso: 25/25 | Artículos: ['B', 'L', 'K', 'C', 'M', 'Cu']
Gen  40 | Mejor valor:  67 | Peso: 25/25 | Artículos: ['B', 'L', 'K', 'C', 'M', 'Cu']
Gen  50 | Mejor valor:  67 | Peso: 25/25 | Artículos: ['B', 'L', 'K', 'C', 'M', 'Cu']
Gen  60 | Mejor valor:  67 | Peso: 25/25 | Artículos: ['B', 'L', 'K', 'C', 'M', 'Cu']
Gen  70 | Mejor valor:  67 | Peso: 25/25 | Artículos: ['B', 'L', 'K', 'C', 'M', 'Cu']
Gen  80 | Mejor valor:  67 | Peso: 25/25 | Artículos: ['B', 'L', 'K', 'C', 'M', 'Cu']
Gen  90 | Mejor valor:  67 | Peso: 25/25 | Artículos: ['B', 'L', 'K', 'C', 'M', 'Cu']
Gen  99 | Mejor valor:  67 | Peso: 25/25 | Artículos: ['B', 'L', 'K', 'C', 'M', 'Cu']

=== RESULTADO FINAL ===
Cromosoma óptimo: [1, 1, 1, 0, 1, 